# Tree-Based Classification and Ensemble Models

This notebook evaluates tree-based classification approaches for identifying dual JNK3/GSK3β candidates using the preprocessed RDKit descriptors. It trains and evaluates XGBoost and Extra Trees models, performs validation-based threshold selection, and compares them with the earlier baseline classifiers. A soft-voting ensemble is also evaluated, followed by a t-SNE projection to visualise the chemical space of the dataset. Model selection and threshold tuning are performed using the training and validation data.

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.utils.class_weight import compute_sample_weight

from sklearn.metrics import (
    balanced_accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

DATA_FILE = "preprocessed_model_data.csv"
DESCRIPTOR_FILE = "selected_descriptor_names.csv"
RANDOM_SEED = 42

data_df = pd.read_csv(DATA_FILE)

descriptor_columns = pd.read_csv(
    DESCRIPTOR_FILE
)["descriptor"].tolist()

print("Dataset loaded successfully.")
print("Dataset shape:", data_df.shape)
print("Number of descriptors:", len(descriptor_columns))

print("\nMolecules in each split:")
print(data_df["split"].value_counts())

In [ ]:
# Using the existing scaffold-based split

train_df = data_df[
    data_df["split"] == "train"
].copy()

validation_df = data_df[
    data_df["split"] == "validation"
].copy()

# Keeping the test set untouched
test_df = data_df[
    data_df["split"] == "test"
].copy()


# Molecular descriptors used as inputs
X_train = train_df[
    descriptor_columns
]

X_validation = validation_df[
    descriptor_columns
]


# Classification label:
# 1 = dual candidate
# 0 = not a dual candidate
y_train = train_df[
    "dual_candidate"
].astype(int)

y_validation = validation_df[
    "dual_candidate"
].astype(int)


print("Training feature shape:", X_train.shape)
print("Validation feature shape:", X_validation.shape)
print("Untouched test molecules:", len(test_df))

print("\nTraining class counts:")
print(y_train.value_counts())

print("\nValidation class counts:")
print(y_validation.value_counts())

print("\nTraining dual percentage:")
print(round(y_train.mean() * 100, 2))

print("\nValidation dual percentage:")
print(round(y_validation.mean() * 100, 2))

print("\nThe test set has not been used.")

In [ ]:
# Giving more importance to the smaller class during training

training_sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

print(
    "Number of sample weights:",
    len(training_sample_weights)
)

print(
    "\nAverage weight for class 0:",
    round(
        training_sample_weights[
            y_train.values == 0
        ].mean(),
        3
    )
)

print(
    "Average weight for class 1:",
    round(
        training_sample_weights[
            y_train.values == 1
        ].mean(),
        3
    )
)

In [ ]:
# Create the baseline XGBoost classifier

xgboost_model = XGBClassifier(
    objective="binary:logistic",
    
    n_estimators=1000,
    learning_rate=0.05,
    
    max_depth=6,
    min_child_weight=3,
    
    subsample=0.80,
    colsample_bytree=0.80,
    
    reg_alpha=0.10,
    reg_lambda=1.00,
    
    tree_method="hist",
    eval_metric="logloss",
    early_stopping_rounds=50,
    
    random_state=RANDOM_SEED,
    n_jobs=-1
)


print("Training the XGBoost classifier...")

xgboost_model.fit(
    X_train,
    y_train,
    
    sample_weight=training_sample_weights,
    
    eval_set=[
        (X_validation, y_validation)
    ],
    
    verbose=50
)

print("\nXGBoost training completed.")

print(
    "Best boosting iteration:",
    xgboost_model.best_iteration
)

In [ ]:
# Predicting validation probabilities

xgboost_validation_probabilities = (
    xgboost_model.predict_proba(X_validation)[:, 1]
)

# Fixed classification threshold:
# probability >= 0.50 becomes class 1

CLASSIFICATION_THRESHOLD = 0.50

xgboost_validation_predictions = (
    xgboost_validation_probabilities
    >= CLASSIFICATION_THRESHOLD
).astype(int)


# Confusion-matrix values

tn, fp, fn, tp = confusion_matrix(
    y_validation,
    xgboost_validation_predictions
).ravel()

specificity = tn / (tn + fp)


# Store all validation metrics

xgboost_results = {
    "model": "XGBoost",
    "threshold": CLASSIFICATION_THRESHOLD,

    "balanced_accuracy": balanced_accuracy_score(
        y_validation,
        xgboost_validation_predictions
    ),

    "mcc": matthews_corrcoef(
        y_validation,
        xgboost_validation_predictions
    ),

    "roc_auc": roc_auc_score(
        y_validation,
        xgboost_validation_probabilities
    ),

    "average_precision": average_precision_score(
        y_validation,
        xgboost_validation_probabilities
    ),

    "precision": precision_score(
        y_validation,
        xgboost_validation_predictions,
        zero_division=0
    ),

    "recall": recall_score(
        y_validation,
        xgboost_validation_predictions,
        zero_division=0
    ),

    "specificity": specificity,

    "f1_score": f1_score(
        y_validation,
        xgboost_validation_predictions,
        zero_division=0
    ),

    "true_negative": tn,
    "false_positive": fp,
    "false_negative": fn,
    "true_positive": tp
}


xgboost_results_df = pd.DataFrame(
    [xgboost_results]
)

display(
    xgboost_results_df.round(3)
)

print("\nValidation predictions completed.")
print("The test set has not been used.")

In [ ]:
# Evaluate probability thresholds using validation data
threshold_results = []

for threshold in np.arange(0.20, 0.81, 0.01):
    predicted_labels = (
        xgboost_validation_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        predicted_labels
    ).ravel()

    specificity = tn / (tn + fp)

    threshold_results.append({
        "threshold": threshold,
        "balanced_accuracy": balanced_accuracy_score(
            y_validation,
            predicted_labels
        ),
        "mcc": matthews_corrcoef(
            y_validation,
            predicted_labels
        ),
        "precision": precision_score(
            y_validation,
            predicted_labels,
            zero_division=0
        ),
        "recall": recall_score(
            y_validation,
            predicted_labels,
            zero_division=0
        ),
        "specificity": specificity,
        "f1_score": f1_score(
            y_validation,
            predicted_labels,
            zero_division=0
        ),
        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
        "true_positive": tp,
    })

threshold_results_df = pd.DataFrame(threshold_results)

# Select the best threshold using balanced accuracy,
# with MCC used as the tie-breaker
best_threshold_result = (
    threshold_results_df
    .sort_values(
        by=["balanced_accuracy", "mcc"],
        ascending=False
    )
    .iloc[0]
)

print("Best threshold based on balanced accuracy:")
display(best_threshold_result.to_frame().T.round(3))

print("\nTop 10 thresholds:")
display(
    threshold_results_df
    .sort_values(
        by=["balanced_accuracy", "mcc"],
        ascending=False
    )
    .head(10)
    .round(3)
)

In [ ]:
# Comparing the best thresholds according to different metrics

best_balanced_accuracy = (
    threshold_results_df
    .sort_values(
        by="balanced_accuracy",
        ascending=False
    )
    .iloc[0]
)

best_mcc = (
    threshold_results_df
    .sort_values(
        by="mcc",
        ascending=False
    )
    .iloc[0]
)

best_f1 = (
    threshold_results_df
    .sort_values(
        by="f1_score",
        ascending=False
    )
    .iloc[0]
)

best_thresholds_df = pd.DataFrame([
    {
        "selection_metric": "Balanced accuracy",
        **best_balanced_accuracy.to_dict()
    },
    {
        "selection_metric": "MCC",
        **best_mcc.to_dict()
    },
    {
        "selection_metric": "F1 score",
        **best_f1.to_dict()
    }
])

display(best_thresholds_df.round(3))

print("The test set has not been used.")

In [ ]:
# Using the validation-selected threshold

FINAL_VALIDATION_THRESHOLD = 0.70

final_xgboost_predictions = (
    xgboost_validation_probabilities
    >= FINAL_VALIDATION_THRESHOLD
).astype(int)


# Calculating final validation confusion matrix

tn, fp, fn, tp = confusion_matrix(
    y_validation,
    final_xgboost_predictions
).ravel()

specificity = tn / (tn + fp)


final_xgboost_results = pd.DataFrame([{
    "model": "XGBoost",
    "threshold": FINAL_VALIDATION_THRESHOLD,

    "balanced_accuracy": balanced_accuracy_score(
        y_validation,
        final_xgboost_predictions
    ),

    "mcc": matthews_corrcoef(
        y_validation,
        final_xgboost_predictions
    ),

    "roc_auc": roc_auc_score(
        y_validation,
        xgboost_validation_probabilities
    ),

    "average_precision": average_precision_score(
        y_validation,
        xgboost_validation_probabilities
    ),

    "precision": precision_score(
        y_validation,
        final_xgboost_predictions,
        zero_division=0
    ),

    "recall": recall_score(
        y_validation,
        final_xgboost_predictions,
        zero_division=0
    ),

    "specificity": specificity,

    "f1_score": f1_score(
        y_validation,
        final_xgboost_predictions,
        zero_division=0
    ),

    "true_negative": tn,
    "false_positive": fp,
    "false_negative": fn,
    "true_positive": tp
}])


xgboost_predictions_df = pd.DataFrame({
    "canonical_smiles": validation_df[
        "canonical_smiles"
    ].values,

    "true_label": y_validation.values,

    "predicted_probability": (
        xgboost_validation_probabilities
    ),

    "predicted_label": final_xgboost_predictions,

    "threshold": FINAL_VALIDATION_THRESHOLD
})


final_xgboost_results.to_csv(
    "xgboost_validation_results.csv",
    index=False
)

xgboost_predictions_df.to_csv(
    "xgboost_validation_predictions.csv",
    index=False
)

threshold_results_df.to_csv(
    "xgboost_threshold_results.csv",
    index=False
)

joblib.dump(
    xgboost_model,
    "xgboost_classification_model.joblib"
)


display(final_xgboost_results.round(3))

print("Saved: xgboost_validation_results.csv")
print("Saved: xgboost_validation_predictions.csv")
print("Saved: xgboost_threshold_results.csv")
print("Saved: xgboost_classification_model.joblib")
print("\nThe test set has not been used.")

In [ ]:
# Loading previous classification results

previous_results_df = pd.read_csv(
    "classification_validation_results.csv"
)

# The previous classifiers used the default threshold of 0.50
if "threshold" not in previous_results_df.columns:
    previous_results_df.insert(
        1,
        "threshold",
        0.50
    )


# Keep both XGBoost results:
# 1. Default threshold for a fair model comparison
# 2. Tuned threshold for the best balanced accuracy

xgboost_default_comparison = (
    xgboost_results_df.copy()
)

xgboost_default_comparison["model"] = (
    "XGBoost - default threshold"
)

xgboost_tuned_comparison = (
    final_xgboost_results.copy()
)

xgboost_tuned_comparison["model"] = (
    "XGBoost - tuned threshold"
)


comparison_columns = [
    "model",
    "threshold",
    "balanced_accuracy",
    "mcc",
    "roc_auc",
    "average_precision",
    "precision",
    "recall",
    "specificity",
    "f1_score",
    "true_negative",
    "false_positive",
    "false_negative",
    "true_positive"
]


all_classifier_results_df = pd.concat(
    [
        previous_results_df[comparison_columns],
        xgboost_default_comparison[comparison_columns],
        xgboost_tuned_comparison[comparison_columns]
    ],
    ignore_index=True
)


all_classifier_results_df = (
    all_classifier_results_df
    .sort_values(
        by=[
            "balanced_accuracy",
            "mcc"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    all_classifier_results_df.round(3)
)


all_classifier_results_df.to_csv(
    "all_classification_validation_results.csv",
    index=False
)


print(
    "Saved: all_classification_validation_results.csv"
)

print(
    "\nThe test set has not been used."
)

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

extra_trees_model = ExtraTreesClassifier(
    n_estimators=500,
    max_features="sqrt",
    class_weight="balanced",
    random_state=RANDOM_SEED,
    n_jobs=-1
)

print("Training Extra Trees classifier...")

extra_trees_model.fit(
    X_train,
    y_train
)

print("Extra Trees training completed.")

In [ ]:
# Predicting validation probabilities
extra_trees_probabilities = (
    extra_trees_model.predict_proba(X_validation)[:, 1]
)

EXTRA_TREES_THRESHOLD = 0.50

extra_trees_predictions = (
    extra_trees_probabilities >= EXTRA_TREES_THRESHOLD
).astype(int)


# Confusion matrix
tn, fp, fn, tp = confusion_matrix(
    y_validation,
    extra_trees_predictions
).ravel()

specificity = tn / (tn + fp)


# Storing results
extra_trees_results = {
    "model": "Extra Trees",
    "threshold": EXTRA_TREES_THRESHOLD,

    "balanced_accuracy": balanced_accuracy_score(
        y_validation,
        extra_trees_predictions
    ),

    "mcc": matthews_corrcoef(
        y_validation,
        extra_trees_predictions
    ),

    "roc_auc": roc_auc_score(
        y_validation,
        extra_trees_probabilities
    ),

    "average_precision": average_precision_score(
        y_validation,
        extra_trees_probabilities
    ),

    "precision": precision_score(
        y_validation,
        extra_trees_predictions,
        zero_division=0
    ),

    "recall": recall_score(
        y_validation,
        extra_trees_predictions,
        zero_division=0
    ),

    "specificity": specificity,

    "f1_score": f1_score(
        y_validation,
        extra_trees_predictions,
        zero_division=0
    ),

    "true_negative": tn,
    "false_positive": fp,
    "false_negative": fn,
    "true_positive": tp
}


extra_trees_results_df = pd.DataFrame(
    [extra_trees_results]
)

display(
    extra_trees_results_df.round(3)
)

print("\nThe test set has not been used.")

In [ ]:
# Test different Extra Trees probability thresholds

extra_trees_threshold_results = []

for threshold in np.arange(0.20, 0.81, 0.01):

    predicted_labels = (
        extra_trees_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        predicted_labels
    ).ravel()

    specificity = tn / (tn + fp)

    extra_trees_threshold_results.append({
        "threshold": threshold,

        "balanced_accuracy": balanced_accuracy_score(
            y_validation,
            predicted_labels
        ),

        "mcc": matthews_corrcoef(
            y_validation,
            predicted_labels
        ),

        "precision": precision_score(
            y_validation,
            predicted_labels,
            zero_division=0
        ),

        "recall": recall_score(
            y_validation,
            predicted_labels,
            zero_division=0
        ),

        "specificity": specificity,

        "f1_score": f1_score(
            y_validation,
            predicted_labels,
            zero_division=0
        ),

        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
        "true_positive": tp
    })


extra_trees_threshold_results_df = pd.DataFrame(
    extra_trees_threshold_results
)


print("Best Extra Trees threshold based on balanced accuracy:")

display(
    extra_trees_threshold_results_df
    .sort_values(
        by=["balanced_accuracy", "mcc"],
        ascending=False
    )
    .head(10)
    .round(3)
)

print("\nThe test set has not been used.")

In [ ]:
# Using the validation-selected Extra Trees threshold
FINAL_EXTRA_TREES_THRESHOLD = 0.79

final_extra_trees_predictions = (
    extra_trees_probabilities >= FINAL_EXTRA_TREES_THRESHOLD
).astype(int)

# Calculating final validation confusion matrix
tn, fp, fn, tp = confusion_matrix(
    y_validation,
    final_extra_trees_predictions
).ravel()

specificity = tn / (tn + fp)

# Storing final validation metrics
final_extra_trees_results = pd.DataFrame([{
    "model": "Extra Trees - tuned threshold",
    "threshold": FINAL_EXTRA_TREES_THRESHOLD,
    "balanced_accuracy": balanced_accuracy_score(
        y_validation,
        final_extra_trees_predictions
    ),
    "mcc": matthews_corrcoef(
        y_validation,
        final_extra_trees_predictions
    ),
    "roc_auc": roc_auc_score(
        y_validation,
        extra_trees_probabilities
    ),
    "average_precision": average_precision_score(
        y_validation,
        extra_trees_probabilities
    ),
    "precision": precision_score(
        y_validation,
        final_extra_trees_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_validation,
        final_extra_trees_predictions,
        zero_division=0
    ),
    "specificity": specificity,
    "f1_score": f1_score(
        y_validation,
        final_extra_trees_predictions,
        zero_division=0
    ),
    "true_negative": tn,
    "false_positive": fp,
    "false_negative": fn,
    "true_positive": tp,
}])

# Saving molecule-level validation predictions
extra_trees_predictions_df = pd.DataFrame({
    "canonical_smiles": validation_df["canonical_smiles"].values,
    "true_label": y_validation.values,
    "predicted_probability": extra_trees_probabilities,
    "predicted_label": final_extra_trees_predictions,
    "threshold": FINAL_EXTRA_TREES_THRESHOLD,
})

# Saving outputs
final_extra_trees_results.to_csv(
    "extra_trees_validation_results.csv",
    index=False
)

extra_trees_predictions_df.to_csv(
    "extra_trees_validation_predictions.csv",
    index=False
)

extra_trees_threshold_results_df.to_csv(
    "extra_trees_threshold_results.csv",
    index=False
)

joblib.dump(
    extra_trees_model,
    "extra_trees_classification_model.joblib"
)

display(final_extra_trees_results.round(3))

print("Saved: extra_trees_validation_results.csv")
print("Saved: extra_trees_validation_predictions.csv")
print("Saved: extra_trees_threshold_results.csv")
print("Saved: extra_trees_classification_model.joblib")
print("\nThe test set has not been used.")

In [ ]:
# Loading the comparison containing previous models and XGBoost

current_comparison_df = pd.read_csv(
    "all_classification_validation_results.csv"
)


# Preparing Extra Trees results at the default threshold

extra_trees_default_comparison = (
    extra_trees_results_df.copy()
)

extra_trees_default_comparison["model"] = (
    "Extra Trees - default threshold"
)


# The tuned result already has the correct model name

extra_trees_tuned_comparison = (
    final_extra_trees_results.copy()
)


comparison_columns = [
    "model",
    "threshold",
    "balanced_accuracy",
    "mcc",
    "roc_auc",
    "average_precision",
    "precision",
    "recall",
    "specificity",
    "f1_score",
    "true_negative",
    "false_positive",
    "false_negative",
    "true_positive"
]


updated_classifier_comparison_df = pd.concat(
    [
        current_comparison_df[comparison_columns],
        extra_trees_default_comparison[comparison_columns],
        extra_trees_tuned_comparison[comparison_columns]
    ],
    ignore_index=True
)


# Removing accidental duplicate rows if this cell is run again

updated_classifier_comparison_df = (
    updated_classifier_comparison_df
    .drop_duplicates(
        subset=["model", "threshold"],
        keep="last"
    )
    .sort_values(
        by=["balanced_accuracy", "mcc"],
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    updated_classifier_comparison_df.round(3)
)


updated_classifier_comparison_df.to_csv(
    "all_classification_validation_results_updated.csv",
    index=False
)


print(
    "Saved: all_classification_validation_results_updated.csv"
)

print("\nThe test set has not been used.")

In [ ]:
# Check available validation-prediction files and their columns

previous_predictions_df = pd.read_csv(
    "classification_validation_predictions.csv"
)

xgboost_predictions_saved_df = pd.read_csv(
    "xgboost_validation_predictions.csv"
)

extra_trees_predictions_saved_df = pd.read_csv(
    "extra_trees_validation_predictions.csv"
)


print("Previous classifier prediction columns:")
print(previous_predictions_df.columns.tolist())

print("\nXGBoost prediction columns:")
print(xgboost_predictions_saved_df.columns.tolist())

print("\nExtra Trees prediction columns:")
print(extra_trees_predictions_saved_df.columns.tolist())


print("\nPrevious prediction shape:")
print(previous_predictions_df.shape)

print("\nXGBoost prediction shape:")
print(xgboost_predictions_saved_df.shape)

print("\nExtra Trees prediction shape:")
print(extra_trees_predictions_saved_df.shape)

In [ ]:
# Converting previous classifier predictions
# from long format into one probability column per model

previous_probabilities_wide = (
    previous_predictions_df
    .pivot(
        index=[
            "canonical_smiles",
            "true_label"
        ],
        columns="model",
        values="predicted_probability"
    )
    .reset_index()
)

# Removing the automatic column-label name
previous_probabilities_wide.columns.name = None


# Preparing XGBoost probabilities

xgboost_probability_table = (
    xgboost_predictions_saved_df[
        [
            "canonical_smiles",
            "true_label",
            "predicted_probability"
        ]
    ]
    .rename(
        columns={
            "predicted_probability":
            "XGBoost"
        }
    )
)


# Preparing Extra Trees probabilities

extra_trees_probability_table = (
    extra_trees_predictions_saved_df[
        [
            "canonical_smiles",
            "true_label",
            "predicted_probability"
        ]
    ]
    .rename(
        columns={
            "predicted_probability":
            "Extra Trees"
        }
    )
)


# Merge all model probabilities

ensemble_probabilities_df = (
    previous_probabilities_wide
    .merge(
        xgboost_probability_table,
        on=[
            "canonical_smiles",
            "true_label"
        ],
        how="inner"
    )
    .merge(
        extra_trees_probability_table,
        on=[
            "canonical_smiles",
            "true_label"
        ],
        how="inner"
    )
)


print(
    "Combined probability table shape:",
    ensemble_probabilities_df.shape
)

print("\nAvailable model columns:")
print(
    ensemble_probabilities_df.columns.tolist()
)

print("\nMissing values:")
print(
    ensemble_probabilities_df.isna().sum()
)

display(
    ensemble_probabilities_df.head()
)

print("\nThe test set has not been used.")

In [ ]:
# Creating an equal-weight soft-voting ensemble

ensemble_model_columns = [
    "Gradient Boosting",
    "Logistic Regression",
    "XGBoost"
]

ensemble_probabilities_df[
    "Soft Voting Probability"
] = ensemble_probabilities_df[
    ensemble_model_columns
].mean(axis=1)


ENSEMBLE_THRESHOLD = 0.50

ensemble_predictions = (
    ensemble_probabilities_df[
        "Soft Voting Probability"
    ] >= ENSEMBLE_THRESHOLD
).astype(int)


ensemble_true_labels = (
    ensemble_probabilities_df[
        "true_label"
    ].astype(int)
)


tn, fp, fn, tp = confusion_matrix(
    ensemble_true_labels,
    ensemble_predictions
).ravel()

specificity = tn / (tn + fp)


soft_voting_results = pd.DataFrame([{
    "model": "Soft Voting: GB + LR + XGBoost",
    "threshold": ENSEMBLE_THRESHOLD,

    "balanced_accuracy": balanced_accuracy_score(
        ensemble_true_labels,
        ensemble_predictions
    ),

    "mcc": matthews_corrcoef(
        ensemble_true_labels,
        ensemble_predictions
    ),

    "roc_auc": roc_auc_score(
        ensemble_true_labels,
        ensemble_probabilities_df[
            "Soft Voting Probability"
        ]
    ),

    "average_precision": average_precision_score(
        ensemble_true_labels,
        ensemble_probabilities_df[
            "Soft Voting Probability"
        ]
    ),

    "precision": precision_score(
        ensemble_true_labels,
        ensemble_predictions,
        zero_division=0
    ),

    "recall": recall_score(
        ensemble_true_labels,
        ensemble_predictions,
        zero_division=0
    ),

    "specificity": specificity,

    "f1_score": f1_score(
        ensemble_true_labels,
        ensemble_predictions,
        zero_division=0
    ),

    "true_negative": tn,
    "false_positive": fp,
    "false_negative": fn,
    "true_positive": tp
}])


display(
    soft_voting_results.round(3)
)

print("\nThe test set has not been used.")

In [ ]:
# Test different thresholds for the soft-voting ensemble

soft_voting_threshold_results = []

ensemble_scores = ensemble_probabilities_df[
    "Soft Voting Probability"
]

true_labels = ensemble_probabilities_df[
    "true_label"
].astype(int)


for threshold in np.arange(0.20, 0.81, 0.01):

    predicted_labels = (
        ensemble_scores >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        true_labels,
        predicted_labels
    ).ravel()

    specificity = tn / (tn + fp)

    soft_voting_threshold_results.append({
        "threshold": threshold,

        "balanced_accuracy": balanced_accuracy_score(
            true_labels,
            predicted_labels
        ),

        "mcc": matthews_corrcoef(
            true_labels,
            predicted_labels
        ),

        "precision": precision_score(
            true_labels,
            predicted_labels,
            zero_division=0
        ),

        "recall": recall_score(
            true_labels,
            predicted_labels,
            zero_division=0
        ),

        "specificity": specificity,

        "f1_score": f1_score(
            true_labels,
            predicted_labels,
            zero_division=0
        ),

        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
        "true_positive": tp
    })


soft_voting_threshold_results_df = pd.DataFrame(
    soft_voting_threshold_results
)


best_soft_voting_thresholds = pd.DataFrame([
    {
        "selection_metric": "Balanced accuracy",
        **soft_voting_threshold_results_df
        .sort_values(
            by="balanced_accuracy",
            ascending=False
        )
        .iloc[0]
        .to_dict()
    },

    {
        "selection_metric": "MCC",
        **soft_voting_threshold_results_df
        .sort_values(
            by="mcc",
            ascending=False
        )
        .iloc[0]
        .to_dict()
    },

    {
        "selection_metric": "F1 score",
        **soft_voting_threshold_results_df
        .sort_values(
            by="f1_score",
            ascending=False
        )
        .iloc[0]
        .to_dict()
    }
])


display(
    best_soft_voting_thresholds.round(3)
)

print("\nThe test set has not been used.")

In [ ]:
# Save the soft-voting ensemble at:
# 0.60 = best balanced accuracy
# 0.40 = best MCC

ensemble_thresholds_to_save = {
    "Soft Voting - best balanced accuracy": 0.60,
    "Soft Voting - best MCC": 0.40
}

saved_ensemble_results = []
saved_ensemble_predictions = []

for model_name, threshold in ensemble_thresholds_to_save.items():

    predicted_labels = (
        ensemble_scores >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        true_labels,
        predicted_labels
    ).ravel()

    specificity = tn / (tn + fp)

    saved_ensemble_results.append({
        "model": model_name,
        "threshold": threshold,

        "balanced_accuracy": balanced_accuracy_score(
            true_labels,
            predicted_labels
        ),

        "mcc": matthews_corrcoef(
            true_labels,
            predicted_labels
        ),

        "roc_auc": roc_auc_score(
            true_labels,
            ensemble_scores
        ),

        "average_precision": average_precision_score(
            true_labels,
            ensemble_scores
        ),

        "precision": precision_score(
            true_labels,
            predicted_labels,
            zero_division=0
        ),

        "recall": recall_score(
            true_labels,
            predicted_labels,
            zero_division=0
        ),

        "specificity": specificity,

        "f1_score": f1_score(
            true_labels,
            predicted_labels,
            zero_division=0
        ),

        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
        "true_positive": tp
    })

    prediction_table = pd.DataFrame({
        "canonical_smiles": ensemble_probabilities_df[
            "canonical_smiles"
        ].values,

        "true_label": true_labels.values,

        "model": model_name,

        "predicted_probability": ensemble_scores.values,

        "predicted_label": predicted_labels.values,

        "threshold": threshold
    })

    saved_ensemble_predictions.append(
        prediction_table
    )


final_soft_voting_results_df = pd.DataFrame(
    saved_ensemble_results
)

final_soft_voting_predictions_df = pd.concat(
    saved_ensemble_predictions,
    ignore_index=True
)


final_soft_voting_results_df.to_csv(
    "soft_voting_validation_results.csv",
    index=False
)

final_soft_voting_predictions_df.to_csv(
    "soft_voting_validation_predictions.csv",
    index=False
)

soft_voting_threshold_results_df.to_csv(
    "soft_voting_threshold_results.csv",
    index=False
)


display(
    final_soft_voting_results_df.round(3)
)

print("Saved: soft_voting_validation_results.csv")
print("Saved: soft_voting_validation_predictions.csv")
print("Saved: soft_voting_threshold_results.csv")
print("\nThe test set has not been used.")

In [ ]:
# Load the comparison containing all individual classifiers

individual_models_df = pd.read_csv(
    "all_classification_validation_results_updated.csv"
)

# Load the two saved soft-voting results

ensemble_models_df = pd.read_csv(
    "soft_voting_validation_results.csv"
)


comparison_columns = [
    "model",
    "threshold",
    "balanced_accuracy",
    "mcc",
    "roc_auc",
    "average_precision",
    "precision",
    "recall",
    "specificity",
    "f1_score",
    "true_negative",
    "false_positive",
    "false_negative",
    "true_positive"
]


final_validation_comparison_df = pd.concat(
    [
        individual_models_df[comparison_columns],
        ensemble_models_df[comparison_columns]
    ],
    ignore_index=True
)


# Remove duplicates if the cell is accidentally run again

final_validation_comparison_df = (
    final_validation_comparison_df
    .drop_duplicates(
        subset=["model", "threshold"],
        keep="last"
    )
    .sort_values(
        by=["balanced_accuracy", "mcc"],
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    final_validation_comparison_df.round(3)
)


final_validation_comparison_df.to_csv(
    "final_classification_validation_comparison.csv",
    index=False
)


print(
    "Saved: final_classification_validation_comparison.csv"
)

print("\nBest balanced-accuracy model:")

display(
    final_validation_comparison_df
    .sort_values(
        by="balanced_accuracy",
        ascending=False
    )
    .head(1)
    .round(3)
)


print("\nBest MCC model:")

display(
    final_validation_comparison_df
    .sort_values(
        by="mcc",
        ascending=False
    )
    .head(1)
    .round(3)
)


print("\nThe test set has not been used.")

In [ ]:
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.manifold import TSNE

RANDOM_SEED = 42
SAMPLE_PER_SPLIT = 2500

data_df = pd.read_csv("preprocessed_model_data.csv")

print("Full dataset shape:", data_df.shape)
print(data_df["split"].value_counts())

# take a random sample from each split so t-sne 
sampled_parts = []

for split_name in ["train", "validation", "test"]:

    split_df = data_df[data_df["split"] == split_name]

    sample_size = min(SAMPLE_PER_SPLIT, len(split_df))

    sampled = split_df.sample(
        n=sample_size,
        random_state=RANDOM_SEED
    )

    sampled_parts.append(sampled)

sample_df = pd.concat(sampled_parts).reset_index(drop=True)

print("\nSampled dataset shape:", sample_df.shape)
print(sample_df["split"].value_counts())

# set up the ecfp6 fingerprint generator (current rdkit api, radius 3 = ecfp6)
fingerprint_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=3,
    fpSize=2048
)

# calculate ecfp6 fingerprints for the sampled molecules
fingerprints = []
valid_rows = []

for i, smiles in enumerate(sample_df["canonical_smiles"]):

    molecule = Chem.MolFromSmiles(smiles)

    if molecule is None:
        continue

    fingerprint = fingerprint_generator.GetFingerprintAsNumPy(molecule)

    fingerprints.append(fingerprint)
    valid_rows.append(i)

    if (i + 1) % 1000 == 0:
        print(f"Processed {i + 1} of {len(sample_df)} molecules")

fingerprint_array = np.array(fingerprints)
sample_df = sample_df.iloc[valid_rows].reset_index(drop=True)

print("\nFingerprint array shape:", fingerprint_array.shape)

# run t-sne on the fingerprints
tsne = TSNE(
    n_components=2,
    init="pca",
    random_state=RANDOM_SEED,
    perplexity=30
)

tsne_coordinates = tsne.fit_transform(fingerprint_array)

sample_df["tsne_1"] = tsne_coordinates[:, 0]
sample_df["tsne_2"] = tsne_coordinates[:, 1]

output_columns = ["canonical_smiles", "split", "dual_candidate", "tsne_1", "tsne_2"]

sample_df[output_columns].to_csv(
    "chemical_space_tsne_coordinates.csv",
    index=False
)

print("\nSaved: chemical_space_tsne_coordinates.csv")
print("Rows:", len(sample_df))